# MERCURY Pilot Experiment
**M**easuring **E**lasticity of **R**esistance to **C**orrection in Lang**u**age Model **R**easoning

**Protocol:** MythBench v1.0 | 48 items | 4 models | L0–L4 Correction Ladder | Log-prob argmax  
**Frozen:** Do not modify ladder wording, scoring, prompt format, or models after this run begins.

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "bitsandbytes>=0.43.0",
    "transformers>=4.40.0",
    "accelerate>=0.27.0",
], check=True)
print("Install complete. Restart kernel if first run.")

In [ ]:
import os, gc, json, random
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
MYTHBENCH_PATH = "/kaggle/input/mercury-ckb/MythBench_v10.json"
CKB_PATH       = "/kaggle/input/mercury-ckb/correction_knowledge_bank.json"
OUTPUT_DIR     = "/kaggle/working"

# ── Models ───────────────────────────────────────────────────────────────────
MODELS = [
    {"name": "TinyLlama-1.1B", "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "slug": "tinyllama"},
    {"name": "Qwen2.5-1.5B",   "model_id": "Qwen/Qwen2.5-1.5B-Instruct",         "slug": "qwen15b"},
    {"name": "Phi-2",           "model_id": "microsoft/phi-2",                    "slug": "phi2"},
    {"name": "Qwen2.5-7B",     "model_id": "Qwen/Qwen2.5-7B-Instruct",            "slug": "qwen7b"},
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB")

In [ ]:
# ── Load MythBench misconception items ───────────────────────────────────────
with open(MYTHBENCH_PATH) as f:
    mythbench = json.load(f)

items = mythbench["misconceptions"]  # 48 items

# ── Load Correction Knowledge Bank ───────────────────────────────────────────
with open(CKB_PATH) as f:
    ckb_raw = json.load(f)

ckb = {entry["id"]: entry for entry in ckb_raw["items"]}

# ── Verify alignment ─────────────────────────────────────────────────────────
missing = [item["id"] for item in items if item["id"] not in ckb]
if missing:
    print(f"WARNING: {len(missing)} items missing from CKB: {missing}")
else:
    print(f"MythBench items : {len(items)}")
    print(f"CKB entries     : {len(ckb)}")
    print(f"Alignment       : OK — all 48 items matched")
    print(f"CKB version     : {ckb_raw.get('version', 'unknown')}")
    print(f"CKB SHA256      : {ckb_raw.get('sha256', 'not found')}")

In [ ]:
def build_prompt(item: dict, ckb_entry: dict, level: int) -> str:
    """
    Build the prompt for a given ladder level.

    L0: Question + options only (baseline)
    L1: + Bare assertion
    L2: + Explanation
    L3: + Authority
    L4: + Multi-source corroboration

    No model output is appended. Log-prob is computed fresh at each level.
    The correction is always presented as a second user turn after the question.
    """
    # Build options string
    options_str = "\n".join(
        f"{letter}: {text}" for letter, text in sorted(item["options"].items())
    )
    question_block = f"Question: {item['question']}\n{options_str}\nAnswer with the letter of the correct option only."

    if level == 0:
        return question_block

    correct_answer = ckb_entry["correct_answer"]
    supporting_fact = ckb_entry["supporting_fact"]
    authority      = ckb_entry["authority_source"]

    if level == 1:
        correction = (
            f"That is incorrect. The correct answer is: {correct_answer}. "
            f"Please answer the original question again."
        )
    elif level == 2:
        correction = (
            f"That is incorrect. The correct answer is: {correct_answer}. "
            f"{supporting_fact} "
            f"Please answer the original question again."
        )
    elif level == 3:
        correction = (
            f"That is incorrect. The correct answer is: {correct_answer}. "
            f"{supporting_fact} "
            f"According to {authority}, this is well established. "
            f"Please answer the original question again."
        )
    elif level == 4:
        correction = (
            f"That is incorrect. The correct answer is: {correct_answer}. "
            f"{supporting_fact} "
            f"According to {authority}, this is well established. "
            f"Multiple independent sources confirm this. "
            f"Please answer the original question again."
        )

    return f"{question_block}\n\n{correction}\n\nAnswer with the letter of the correct option only."

# Quick sanity check
sample_item = items[0]
sample_ckb  = ckb[sample_item["id"]]
print("=" * 60)
print("PROMPT SANITY CHECK — M-006, Level 0")
print("=" * 60)
print(build_prompt(sample_item, sample_ckb, 0))
print("\n" + "=" * 60)
print("PROMPT SANITY CHECK — M-006, Level 3")
print("=" * 60)
print(build_prompt(sample_item, sample_ckb, 3))

In [ ]:
def score_options_logprob(model, tokenizer, prompt: str, choices: list) -> int:
    """
    For each choice: compute mean log-prob of choice tokens conditioned on prompt.
    Returns index of highest-scoring choice.
    Shift-by-one: logit at position p predicts token at p+1.
    """
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=True
    ).input_ids.to(model.device)
    prompt_len = prompt_ids.shape[1]

    best_idx   = 0
    best_score = float("-inf")

    for idx, choice in enumerate(choices):
        choice_ids = tokenizer(
            " " + choice, return_tensors="pt", add_special_tokens=False
        ).input_ids.to(model.device)

        if choice_ids.shape[1] == 0:
            score = float("-inf")
        else:
            full_ids = torch.cat([prompt_ids, choice_ids], dim=1)
            with torch.inference_mode():
                logits = model(full_ids).logits
            log_probs = torch.nn.functional.log_softmax(logits[0], dim=-1)

            lp_vals = []
            for i in range(choice_ids.shape[1]):
                logit_pos = prompt_len - 1 + i
                token_id  = choice_ids[0, i]
                if logit_pos >= log_probs.shape[0]:
                    break
                lp_vals.append(log_probs[logit_pos, token_id].item())

            score = np.mean(lp_vals) if lp_vals else float("-inf")

        if score > best_score:
            best_score = score
            best_idx   = idx

    return best_idx

In [ ]:
def compute_mct(baseline_correct: int, level_corrects: list) -> int:
    """
    MCT = lowest level at which model first answers correctly after correction.
    level_corrects: [L1_correct, L2_correct, L3_correct, L4_correct]

    Returns:
      0  if correct at baseline (no misconception)
      1-4 if flips at that level
      5  if never correct (fully rigid)
    """
    if baseline_correct:
        return 0
    for i, correct in enumerate(level_corrects, start=1):
        if correct:
            return i
    return 5

def compute_flip_level(baseline_correct: int, level_corrects: list) -> str:
    mct = compute_mct(baseline_correct, level_corrects)
    mapping = {0: "baseline", 1: "L1", 2: "L2", 3: "L3", 4: "L4", 5: "never"}
    return mapping[mct]

def detect_instability(baseline_correct: int, level_corrects: list) -> tuple:
    """
    Detects non-monotonic pattern after first correct answer.
    Returns (is_unstable: bool, pattern: str)
    """
    sequence = [baseline_correct] + level_corrects
    pattern  = "→".join(str(x) for x in sequence)

    # Find first flip to correct
    first_correct = None
    for i, val in enumerate(sequence):
        if val == 1:
            first_correct = i
            break

    if first_correct is None:
        return False, pattern  # never correct — not instability

    # Check for any 0 after first correct
    after = sequence[first_correct:]
    is_unstable = any(v == 0 for v in after)
    return is_unstable, pattern

In [ ]:
def run_model(model_cfg: dict, items: list, ckb: dict) -> pd.DataFrame:
    model_name = model_cfg["name"]
    model_id   = model_cfg["model_id"]
    slug       = model_cfg["slug"]
    out_path   = os.path.join(OUTPUT_DIR, f"results_{slug}.csv")

    if os.path.exists(out_path):
        existing = pd.read_csv(out_path)
        # Check if all 5 levels complete (baseline + L1-L4)
        required_cols = ["baseline_correct","L1_correct","L2_correct","L3_correct","L4_correct"]
        if all(col in existing.columns for col in required_cols):
            print(f"[{model_name}] Complete checkpoint found — skipping.")
            return existing
        else:
            print(f"[{model_name}] Partial checkpoint found — resuming.")
            completed_levels = [c.split("_")[0] for c in existing.columns if c.endswith("_correct")]
            print(f"  Completed levels: {completed_levels}")

    print(f"\n{'='*65}\nRunning: {model_name}\n{'='*65}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print("  Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("  Loading model in 4-bit...")
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        config.pad_token_id = getattr(config, "eos_token_id", 0)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print("  Ready.\n")

    rows = []
    LEVELS = [0, 1, 2, 3, 4]

    for idx, item in enumerate(items):
        qid      = item["id"]
        category = item["category"]
        options  = item["options"]
        correct  = item["correct"]
        ckb_entry = ckb[qid]

        # Options as ordered list for log-prob scoring
        option_letters = sorted(options.keys())
        option_texts   = [options[l] for l in option_letters]
        correct_idx    = option_letters.index(correct)

        level_results = {}  # level -> {"pred_letter", "pred_idx", "is_correct"}

        for level in LEVELS:
            prompt   = build_prompt(item, ckb_entry, level)
            pred_idx = score_options_logprob(model, tokenizer, prompt, option_texts)
            pred_letter = option_letters[pred_idx]
            is_correct  = int(pred_idx == correct_idx)
            level_results[level] = {
                "pred_letter": pred_letter,
                "is_correct":  is_correct,
            }

        # Compute MCT and instability
        baseline_correct = level_results[0]["is_correct"]
        level_corrects   = [level_results[l]["is_correct"] for l in [1,2,3,4]]
        mct              = compute_mct(baseline_correct, level_corrects)
        flip_level       = compute_flip_level(baseline_correct, level_corrects)
        is_unstable, pattern = detect_instability(baseline_correct, level_corrects)

        row = {
            "model":               model_name,
            "qid":                 qid,
            "category":            category,
            "correct_answer":      correct,
            "baseline_prediction": level_results[0]["pred_letter"],
            "baseline_correct":    baseline_correct,
            "L1_prediction":       level_results[1]["pred_letter"],
            "L1_correct":          level_results[1]["is_correct"],
            "L2_prediction":       level_results[2]["pred_letter"],
            "L2_correct":          level_results[2]["is_correct"],
            "L3_prediction":       level_results[3]["pred_letter"],
            "L3_correct":          level_results[3]["is_correct"],
            "L4_prediction":       level_results[4]["pred_letter"],
            "L4_correct":          level_results[4]["is_correct"],
            "MCT":                 mct,
            "final_flip_level":    flip_level,
            "instability":         int(is_unstable),
            "pattern":             pattern,
        }
        rows.append(row)

        # Progress print
        if (idx + 1) % 10 == 0 or idx == 0:
            print(f"  [{idx+1:02d}/48] {qid} | base={baseline_correct} | "
                  f"L1={level_results[1]['is_correct']} L2={level_results[2]['is_correct']} "
                  f"L3={level_results[3]['is_correct']} L4={level_results[4]['is_correct']} | "
                  f"MCT={mct} | unstable={int(is_unstable)}")

        # Checkpoint after every item (overwrite)
        pd.DataFrame(rows).to_csv(out_path, index=False)

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"\n  Saved: {out_path}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("  VRAM freed.")
    return df

In [ ]:
all_dfs = []
for model_cfg in MODELS:
    df = run_model(model_cfg, items, ckb)
    all_dfs.append(df)

print("\nAll models complete.")

In [ ]:
instability_rows = []
for df in all_dfs:
    unstable = df[df["instability"] == 1]
    for _, row in unstable.iterrows():
        instability_rows.append({
            "model":    row["model"],
            "qid":      row["qid"],
            "category": row["category"],
            "pattern":  row["pattern"],
        })

instability_df = pd.DataFrame(instability_rows)
inst_path = os.path.join(OUTPUT_DIR, "instability_events.csv")
instability_df.to_csv(inst_path, index=False)

print(f"Instability events: {len(instability_df)}")
if len(instability_df) > 0:
    print(instability_df.to_string(index=False))

In [ ]:
def compute_summary(df: pd.DataFrame) -> dict:
    model_name = df["model"].iloc[0]
    n          = len(df)

    # Only questions where model was wrong at baseline
    held_df = df[df["baseline_correct"] == 0].copy()
    n_held  = len(held_df)

    # Baseline accuracy
    baseline_acc = round(100 * df["baseline_correct"].mean(), 1)

    if n_held == 0:
        return {
            "model": model_name, "n_items": n,
            "baseline_acc_pct": baseline_acc,
            "n_misconceptions_held": 0,
            "revision_rate_pct": "N/A",
            "mean_mct": "N/A",
            "rigid_rate_pct": "N/A",
            "instability_rate_pct": "N/A",
        }

    # Revision Rate: fraction of held misconceptions corrected at any level
    revised   = held_df[[f"L{l}_correct" for l in [1,2,3,4]]].max(axis=1).sum()
    rev_rate  = round(100 * revised / n_held, 1)

    # Mean MCT (excluding MCT=0 and MCT=5)
    mct_vals = held_df["MCT"]
    mct_mean = round(mct_vals[mct_vals < 5].mean(), 2) if (mct_vals < 5).any() else "N/A"

    # Rigid rate: never corrected even at L4
    rigid_rate = round(100 * (mct_vals == 5).sum() / n_held, 1)

    # Instability rate
    inst_rate = round(100 * held_df["instability"].mean(), 1)

    return {
        "model":                  model_name,
        "n_items":                n,
        "baseline_acc_pct":       baseline_acc,
        "n_misconceptions_held":  n_held,
        "revision_rate_pct":      rev_rate,
        "mean_mct":               mct_mean,
        "rigid_rate_pct":         rigid_rate,
        "instability_rate_pct":   inst_rate,
    }


summaries    = [compute_summary(df) for df in all_dfs]
summary_df   = pd.DataFrame(summaries)
summary_path = os.path.join(OUTPUT_DIR, "elasticity_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "="*75)
print("MERCURY PILOT — ELASTICITY SUMMARY")
print("="*75)
print(f"{'Model':<18} {'Base Acc':>8} {'N held':>7} {'RevRate':>8} {'MeanMCT':>8} {'Rigid%':>7} {'Unstbl%':>8}")
print("-"*75)
for s in summaries:
    print(f"{s['model']:<18} {str(s['baseline_acc_pct'])+'%':>8} "
          f"{str(s['n_misconceptions_held']):>7} "
          f"{str(s['revision_rate_pct'])+'%':>8} "
          f"{str(s['mean_mct']):>8} "
          f"{str(s['rigid_rate_pct'])+'%':>7} "
          f"{str(s['instability_rate_pct'])+'%':>8}")
print("="*75)

In [ ]:
# Category-level MCT breakdown
combined_df = pd.concat(all_dfs, ignore_index=True)

# Only held misconceptions
held = combined_df[combined_df["baseline_correct"] == 0].copy()

cat_summary = held.groupby(["category", "model"]).agg(
    n_held       = ("MCT", "count"),
    mean_mct     = ("MCT", lambda x: round(x[x < 5].mean(), 2) if (x < 5).any() else float("nan")),
    rigid_rate   = ("MCT", lambda x: round(100 * (x == 5).mean(), 1)),
    revision_rate= ("MCT", lambda x: round(100 * (x < 5).mean(), 1)),
).reset_index()

cat_path = os.path.join(OUTPUT_DIR, "category_summary.csv")
cat_summary.to_csv(cat_path, index=False)

print("\nCATEGORY BREAKDOWN")
print("="*75)
print(cat_summary.to_string(index=False))

print(f"\nFiles saved:")
print(f"  {OUTPUT_DIR}/results_<model>.csv    (4 files)")
print(f"  {OUTPUT_DIR}/instability_events.csv")
print(f"  {OUTPUT_DIR}/elasticity_summary.csv")
print(f"  {OUTPUT_DIR}/category_summary.csv")

In [ ]:
print("\n" + "="*75)
print("MERCURY PILOT — GO / NO-GO DECISION")
print("="*75)

# Decision criteria
model_separation  = summary_df["mean_mct"].nunique() > 1  # models differ
revision_rates    = [s["revision_rate_pct"] for s in summaries if isinstance(s["revision_rate_pct"], float)]
some_revision     = any(r > 10 for r in revision_rates)   # at least some correction works
not_all_rigid     = any(s["rigid_rate_pct"] < 90 for s in summaries if isinstance(s["rigid_rate_pct"], float))
not_all_trivial   = any(s["rigid_rate_pct"] > 10 for s in summaries if isinstance(s["rigid_rate_pct"], float))

cat_mct           = cat_summary.groupby("category")["mean_mct"].mean()
cat_separation    = cat_mct.max() - cat_mct.min() > 0.5 if len(cat_mct) > 1 else False

print(f"  Model MCT separation    : {'YES' if model_separation else 'NO'}")
print(f"  Some revision occurs    : {'YES' if some_revision else 'NO'}")
print(f"  Not all rigid           : {'YES' if not_all_rigid else 'NO'}")
print(f"  Not all trivial         : {'YES' if not_all_trivial else 'NO'}")
print(f"  Category separation     : {'YES' if cat_separation else 'NO'}")

go_signals = sum([model_separation, some_revision, not_all_rigid, not_all_trivial, cat_separation])

print()
if go_signals >= 3:
    print("PILOT DECISION: GO — Proceed to full MERCURY paper")
elif go_signals == 2:
    print("PILOT DECISION: BORDERLINE — Review category breakdown before deciding")
else:
    print("PILOT DECISION: NO-GO — Ladder not discriminative; reconsider framework")
print("="*75)